In [16]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, classification_report
import tensorflow as tf
from tensorflow.keras.models import load_model

warnings.filterwarnings('ignore')

ULKELER     = ['tr', 'fi', 'au']
MODELLER    = ['xgboost', 'randomforest', 'mlp', 'knn', 'rnn']
POLLUTANTS  = ['co', 'no2', 'o3', 'pm10', 'pm25', 'so2']
WINDOW_SIZE = 7 

# HKI (Hava Kalitesi İndeksi) eşikleri
HKI_ESIKLERI = {
    'so2':  [(100, 'iyi'), (250, 'Orta'), (np.inf, 'Kotu')],
    'no2':  [(100, 'iyi'), (200, 'Orta'), (np.inf, 'Kotu')],
    'co':   [(5500, 'iyi'), (10000, 'Orta'), (np.inf, 'Kotu')],
    'o3':   [(120, 'iyi'), (160, 'Orta'), (np.inf, 'Kotu')],
    'pm10': [(50, 'iyi'), (100, 'Orta'), (np.inf, 'Kotu')],
    'pm25': [(12.0, 'iyi'), (35.4, 'Orta'), (np.inf, 'Kotu')]
}
KATEGORI_ETIKETLERI = ['iyi', 'Orta', 'Kotu']

def sayisal_degeri_kategoriye_cevir(degerler, esikler):
    kategoriler = []
    for deger in degerler:
        for sinir, kategori in esikler:
            if deger <= sinir:
                kategoriler.append(kategori)
                break
    return kategoriler

def create_windowed_dataset_for_test(X, seq_feat_count, w):
    Xw, Xs = [], []
    for i in range(len(X) - w + 1):
        Xw.append(X[i:i+w, :seq_feat_count])
        Xs.append(X[i+w-1, seq_feat_count:])
    return np.array(Xw), np.array(Xs)

def main():
    all_sonucs = []
    print("\nTÜM MODELLER İÇİN BİRLEŞİK TEST SÜRECİ BAŞLATILDI\n")

    for country in ULKELER:
        print(f"\n\nÜLKE: {country.upper()} İÇİN TESTLER BAŞLIYOR...")
        for model_name in MODELLER:
            for target in POLLUTANTS:
                if model_name == 'rnn':
                    file_suffix = f"rnn_{target}_{country}"
                    model_path  = f"model_{file_suffix}.h5"
                else:
                    model_file_suffix = f"{model_name}_{target}_{country}"
                    test_data_suffix  = f"{target}_{country}"
                    model_path  = f"model_{model_file_suffix}.pkl"
                
                if not os.path.exists(model_path):
                    continue

                print(f"\n  Test Ediliyor: Model={model_name.upper()}, Kirletici={target.upper()}")

                try:
                    # Dosyaları yükleme
                    if model_name == 'rnn':
                        X_test_df = pd.read_csv(f"X_test_{file_suffix}.csv", index_col='from_date', parse_dates=True)
                        y_test_df = pd.read_csv(f"y_test_{file_suffix}.csv", index_col='from_date', parse_dates=True)
                        x_scaler = joblib.load(f"x_scaler_{file_suffix}.pkl")
                        y_scaler = joblib.load(f"y_scaler_{file_suffix}.pkl")
                    else:
                        X_test_df = pd.read_csv(f"X_test_{test_data_suffix}.csv", index_col='from_date', parse_dates=True)
                        y_test_df = pd.read_csv(f"y_test_{test_data_suffix}.csv", index_col='from_date', parse_dates=True)
                        x_scaler = joblib.load(f"x_scaler_{model_file_suffix}.pkl")
                        y_scaler = joblib.load(f"y_scaler_{model_file_suffix}.pkl")

                    # Tahmin yapma
                    if model_name == 'rnn':
                        model = load_model(model_path, compile=False)
                        seq_feats = joblib.load(f"seq_features_{file_suffix}.pkl")
                        stat_feats = joblib.load(f"stat_features_{file_suffix}.pkl")
                        X_test_s = x_scaler.transform(X_test_df[seq_feats + stat_feats])
                        Xw_te, Xs_te = create_windowed_dataset_for_test(X_test_s, len(seq_feats), WINDOW_SIZE)
                        
                        y_true = y_test_df.iloc[WINDOW_SIZE-1:].values
                        preds_s = model.predict([Xw_te, Xs_te], verbose=0)
                        y_pred = y_scaler.inverse_transform(preds_s)
                    else:
                        model = joblib.load(model_path)
                        selector = joblib.load(f"selector_{model_file_suffix}.pkl")
                        X_aligned = X_test_df.reindex(columns=x_scaler.feature_names_in_, fill_value=0)
                        Xs = x_scaler.transform(X_aligned)
                        Xf = selector.transform(Xs)
                        
                        y_true = y_test_df.values
                        preds_s = model.predict(Xf).reshape(-1, 1)
                        y_pred = y_scaler.inverse_transform(preds_s)
                    
                    # Logaritmik dönüşümü geri alma
                    y_true_final = np.expm1(y_true).flatten()
                    y_pred_final = np.expm1(y_pred).flatten()

                    # 1) REGRESYON METRİKLERİ
                    rmse = np.sqrt(mean_squared_error(y_true_final, y_pred_final))
                    mae = mean_absolute_error(y_true_final, y_pred_final)
                    r2 = r2_score(y_true_final, y_pred_final)
                    print(f"    Regresyon Sonuçları: RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")
                    
                    # 2) SINIFLANDIRMA METRİKLERİ
                    accuracy, f1_kotu = None, None
                    if target in HKI_ESIKLERI:
                        esikler = HKI_ESIKLERI[target]
                        y_true_kategori = sayisal_degeri_kategoriye_cevir(y_true_final, esikler)
                        y_pred_kategori = sayisal_degeri_kategoriye_cevir(y_pred_final, esikler)

                        rapor_dict = classification_report(y_true_kategori, y_pred_kategori, labels=KATEGORI_ETIKETLERI, zero_division=0, output_dict=True)
                        accuracy = rapor_dict.get('accuracy', 0.0)
                        f1_kotu = rapor_dict.get('Kotu', {}).get('f1-score', 0.0)
                        
                        print(f"    Sınıflandırma Sonuçları: Accuracy={accuracy:.4f}, F1 (Kötü+)={f1_kotu:.4f}")

                        # Sonuçları kaydetme
                        output_dir = 'prediction_details'
                        if not os.path.exists(output_dir):
                            os.makedirs(output_dir)
                        
                        # Doğru tarih aralığını alma
                        if model_name == 'rnn':
                            dates = y_test_df.iloc[WINDOW_SIZE-1:].index
                        else:
                            dates = y_test_df.index
                            
                        # Sonuçları içeren bir DataFrame oluşturma
                        prediction_df = pd.DataFrame({
                            'Tarih': dates,
                            'Gercek_Deger': y_true_final,
                            'Tahmin_Deger': y_pred_final,
                            'Gercek_Kategori': y_true_kategori,
                            'Tahmin_Kategori': y_pred_kategori
                        })
                        
                        # DataFrame'i CSV olarak kaydetme
                        prediction_filename = f'{output_dir}/predictions_{country}_{model_name}_{target}.csv'
                        prediction_df.to_csv(prediction_filename, index=False)
                        print(f"    Detaylı tahminler '{prediction_filename}' olarak kaydedildi.")

                    # Tüm sonuçları saklama
                    all_sonucs.append({'Ülke': country.upper(), 'Model': model_name, 'Kirletici': target.upper(),
                                       'RMSE': rmse, 'MAE': mae, 'R2': r2, 
                                       'Accuracy': accuracy, 'f1_kotu_': f1_kotu})
                except Exception as e:
                    print(f"    HATA: {e}")

    if all_sonucs:
        df_sum = pd.DataFrame(all_sonucs)
        summary_filename = 'final_sonuclar_tum_modeller_FULL.csv'
        df_sum.to_csv(summary_filename, index=False)
        print(f"\n[BAŞARILI] Tam özet tablosu '{summary_filename}' olarak kaydedildi.")
    else:
        print("\n[BAŞARISIZ] Hiç sonuç elde edilemedi.")

if __name__ == '__main__':
    main()


TÜM MODELLER İÇİN BİRLEŞİK TEST SÜRECİ BAŞLATILDI



ÜLKE: TR İÇİN TESTLER BAŞLIYOR...

  Test Ediliyor: Model=XGBOOST, Kirletici=CO
    Regresyon Sonuçları: RMSE=19.2217, MAE=19.2217, R²=0.0000
    Sınıflandırma Sonuçları: Accuracy=1.0000, F1 (Kötü+)=0.0000
    Detaylı tahminler 'prediction_details/predictions_tr_xgboost_co.csv' olarak kaydedildi.

  Test Ediliyor: Model=XGBOOST, Kirletici=NO2
    Regresyon Sonuçları: RMSE=23.8370, MAE=19.2805, R²=0.2929
    Sınıflandırma Sonuçları: Accuracy=0.8123, F1 (Kötü+)=0.0000
    Detaylı tahminler 'prediction_details/predictions_tr_xgboost_no2.csv' olarak kaydedildi.

  Test Ediliyor: Model=XGBOOST, Kirletici=O3
    Regresyon Sonuçları: RMSE=10.7858, MAE=9.4246, R²=-0.1739
    Sınıflandırma Sonuçları: Accuracy=1.0000, F1 (Kötü+)=0.0000
    Detaylı tahminler 'prediction_details/predictions_tr_xgboost_o3.csv' olarak kaydedildi.

  Test Ediliyor: Model=XGBOOST, Kirletici=PM10
    Regresyon Sonuçları: RMSE=16.0841, MAE=12.1065, R²=-0.4127
    Sın